In [1]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("SalesSummaryExplain").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
import sys
sys.path.append("../")
from src.gold.sales_summary_gold import transform_sales_summary

In [4]:
orders = spark.read.parquet("../data/silver/orders")
order_items = spark.read.parquet("../data/silver/order_items")
payments = spark.read.parquet("../data/silver/payments")

In [5]:
sales_summary = transform_sales_summary(orders, order_items, payments)

In [6]:
sales_summary.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(7_day_avg_revenue, 'round('avg('total_revenue) windowspecdefinition('order_date ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())), 2), None)]
+- Project [order_date#21, total_orders#44L, total_items_sold#45L, total_revenue#46, total_product_sales#47, total_freight#48, avg_order_value#49, previous_day_revenue#69, revenue_growth#70, cumulative_revenue#71]
   +- Project [order_date#21, total_orders#44L, total_items_sold#45L, total_revenue#46, total_product_sales#47, total_freight#48, avg_order_value#49, previous_day_revenue#69, revenue_growth#70, _we0#73, round(_we0#73, 2) AS cumulative_revenue#71]
      +- Window [sum(total_revenue#46) windowspecdefinition(order_date#21 ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS _we0#73], [order_date#21 ASC NULLS FIRST]
         +- Project [order_date#21, total_orders#44L, total_items_sold#45L, total_revenue#46, total_product_sal